# Stage 2 - judge MAIN (Kaggle **GPU T4 x2 required**, Internet ON)
Add the generate notebook's output (gen_main_seed0.zip and gen_main_seed1.zip) as input. 2 seeds x 8 files x 300 = 4800 judgments at about 2.4 s each = about 3.2 h (extrapolated from the pilot). Exports verdict files plus summary_main.csv and criteria.json (no responses).

In [1]:
import os, glob, subprocess, torch
assert torch.cuda.is_available(), "NO GPU. Settings -> Accelerator -> GPU T4 x2, then restart. (The last attempt ran on CPU and timed out after 12 h.)"
print("GPUs:", torch.cuda.device_count(), [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
TAG = "main"
GEN_ZIPS = sorted(glob.glob("/kaggle/input/**/gen_main_seed*.zip", recursive=True)); print(GEN_ZIPS); assert len(GEN_ZIPS) >= 1, "add the gen_main zips (generate notebook output) as input"
JUDGE_HF = "Qwen/Qwen2.5-7B-Instruct"; JUDGE_ID = "qwen25-7b"      # fp16 split over 2 GPUs; if memory errors: "Qwen/Qwen2.5-3B-Instruct" and JUDGE_ID = "qwen25-3b"
!git clone -q https://github.com/sarrazer24/reins-mini /kaggle/working/reins-mini
%cd /kaggle/working/reins-mini
!mkdir -p runs
for z in GEN_ZIPS: subprocess.run(f"unzip -qo {z} -d runs", shell=True)
!pip install -q -U transformers accelerate
!git clone -q --depth 1 https://github.com/Geralt1020/REINS /tmp/REINS

GPUs: 2 ['Tesla T4', 'Tesla T4']
['/kaggle/input/notebooks/sarrazer/kaggle-10-generate/reins-mini/gen_main_seed0.zip', '/kaggle/input/notebooks/sarrazer/kaggle-10-generate/reins-mini/gen_main_seed1.zip']
/kaggle/working/reins-mini


In [2]:
for d in sorted(glob.glob(f"runs/{TAG}/seed*")):
    print("judging", d, flush=True)
    subprocess.run(f"python -u scripts/11_run_judge.py --run-dir {d} --guise-dir /tmp/REINS/guise --judge-id {JUDGE_ID} --hf-model {JUDGE_HF}", shell=True, check=True)

judging runs/main/seed0


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 339/339 [00:32<00:00, 10.49it/s]


  harm_only.jsonl 1/300  4.2s per judgment
  harm_only.jsonl 2/300  3.3s per judgment
  harm_only.jsonl 3/300  2.9s per judgment
  harm_only.jsonl 5/300  2.7s per judgment
  harm_only.jsonl 10/300  2.5s per judgment
  harm_only.jsonl 15/300  2.5s per judgment
  harm_only.jsonl 20/300  2.5s per judgment
  harm_only.jsonl 25/300  2.4s per judgment
  harm_only.jsonl 30/300  2.4s per judgment
  harm_only.jsonl 35/300  2.5s per judgment
  harm_only.jsonl 40/300  2.4s per judgment
  harm_only.jsonl 45/300  2.4s per judgment
  harm_only.jsonl 50/300  2.4s per judgment
  harm_only.jsonl 55/300  2.4s per judgment
  harm_only.jsonl 60/300  2.4s per judgment
  harm_only.jsonl 65/300  2.4s per judgment
  harm_only.jsonl 70/300  2.4s per judgment
  harm_only.jsonl 75/300  2.4s per judgment
  harm_only.jsonl 80/300  2.4s per judgment
  harm_only.jsonl 85/300  2.4s per judgment
  harm_only.jsonl 90/300  2.4s per judgment
  harm_only.jsonl 95/300  2.4s per judgment
  harm_only.jsonl 100/300  2.4s per 

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 339/339 [00:12<00:00, 26.92it/s]


  harm_only.jsonl 1/300  3.1s per judgment
  harm_only.jsonl 2/300  2.7s per judgment
  harm_only.jsonl 3/300  2.6s per judgment
  harm_only.jsonl 5/300  2.6s per judgment
  harm_only.jsonl 10/300  2.5s per judgment
  harm_only.jsonl 15/300  2.5s per judgment
  harm_only.jsonl 20/300  2.5s per judgment
  harm_only.jsonl 25/300  2.4s per judgment
  harm_only.jsonl 30/300  2.4s per judgment
  harm_only.jsonl 35/300  2.4s per judgment
  harm_only.jsonl 40/300  2.4s per judgment
  harm_only.jsonl 45/300  2.5s per judgment
  harm_only.jsonl 50/300  2.5s per judgment
  harm_only.jsonl 55/300  2.5s per judgment
  harm_only.jsonl 60/300  2.4s per judgment
  harm_only.jsonl 65/300  2.5s per judgment
  harm_only.jsonl 70/300  2.5s per judgment
  harm_only.jsonl 75/300  2.4s per judgment
  harm_only.jsonl 80/300  2.4s per judgment
  harm_only.jsonl 85/300  2.4s per judgment
  harm_only.jsonl 90/300  2.4s per judgment
  harm_only.jsonl 95/300  2.4s per judgment
  harm_only.jsonl 100/300  2.4s per 

In [3]:
# outcome table next to the paper + the pre-registered criteria
subprocess.run(f"python scripts/12_evaluate.py --runs runs/{TAG} --judge-id {JUDGE_ID} --size 2B --out results/summary_{TAG}.csv", shell=True)

method         seed      n  drop    HRR [95% CI]           SRR    OSR     CR   | paper 2B: HRR SRR OSR CR
original       seed0   300     0   66.7 [61.2, 71.8]     2.0   31.3    0.0   |  88.7   1.7   9.7   0.0
random_any     seed0   300     0   68.0 [62.5, 73.0]     1.7   30.3    0.0   |  87.3   1.7  11.0   0.0
random_active  seed0   300     0   67.7 [62.2, 72.7]     2.0   30.3    0.0   | -
harm_only      seed0   300     0   63.3 [57.7, 68.6]     2.3   33.7    0.7   | -
refusal_only   seed0   300     0   22.7 [18.3, 27.7]    45.3   31.3    0.7   | -
reins          seed0   300     0   17.0 [13.2, 21.7]    54.3   26.3    2.3   |  24.8  43.9  18.6  12.8
reins_gate     seed0   300     0   17.3 [13.5, 22.0]    54.3   26.0    2.3   |  25.6  43.2  18.7  12.6
original       seed1   300     0   67.0 [61.5, 72.1]     1.7   31.3    0.0   |  88.7   1.7   9.7   0.0
random_any     seed1   300     0   67.0 [61.5, 72.1]     1.7   31.3    0.0   |  87.3   1.7  11.0   0.0
random_active  seed1   300     0 

CompletedProcess(args='python scripts/12_evaluate.py --runs runs/main --judge-id qwen25-7b --size 2B --out results/summary_main.csv', returncode=0)

In [4]:
!find runs/main -name "*.judged_*.jsonl" -o -name "gate_scores_*.csv" | zip -q judged_{JUDGE_ID}_main.zip -@
!cp results/summary_main.csv results/criteria.json . 2>/dev/null; ls -la judged_*.zip summary_main.csv criteria.json
!rm -rf /tmp/REINS

-rw-r--r-- 1 root root  1263 Nov 19 10:36 criteria.json
-rw-r--r-- 1 root root 62925 Nov 19 10:36 judged_qwen25-7b_main.zip
-rw-r--r-- 1 root root  1609 Nov 19 10:36 summary_main.csv
